<div style="
  background: linear-gradient(135deg, #a8edea, #fed6e3);
  padding: 28px;
  border-radius: 24px;
  text-align: center;
  color: #1a1a2e;
  box-shadow: 0 8px 20px rgba(0,0,0,0.12);
  margin-bottom: 20px;
">
  <h1 style="font-size: 42px; margin-bottom: 8px;">🔬 Blood Cell AI Lab 🩸</h1>
  <h2 style="font-size: 24px; margin-top: 0;">Classifying Cancer Cells with Deep Learning</h2>
  <p style="font-size: 18px; line-height: 1.7;">
    Welcome, future AI researcher! In this project, you will build and compare two deep learning models —
    a <strong>Fully Connected Neural Network (MLP)</strong> and a <strong>Convolutional Neural Network (CNN)</strong> —
    to classify blood cell images for cancer detection.
  </p>
</div>

### Imports

In [5]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, datasets
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 9248
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Define Transforms

In [6]:
from torchvision.models import VGG16_Weights

# Get ImageNet normalization constants from torchvision
imagenet_weights = VGG16_Weights.IMAGENET1K_V1
IMAGENET_MEAN = imagenet_weights.transforms().mean
IMAGENET_STD = imagenet_weights.transforms().std

# Transform for training data (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Transform for test data (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print(f"ImageNet mean: {IMAGENET_MEAN}")
print(f"ImageNet std:  {IMAGENET_STD}")
print("Transforms defined successfully.")

ImageNet mean: [0.485, 0.456, 0.406]
ImageNet std:  [0.229, 0.224, 0.225]
Transforms defined successfully.


<span style="color: #e74c3c; font-size: 28px;"><strong>Why these normalization values?</strong></span>

The mean `[0.485, 0.456, 0.406]` and standard deviation `[0.229, 0.224, 0.225]` are the pre-computed channel-wise statistics of the **ImageNet** dataset (over 1 million images, 3 RGB channels). We import them directly from `torchvision.models.VGG16_Weights` to avoid magic numbers.

Using these widely-accepted values ensures our pixel distributions are centered around zero with unit variance, which leads to **faster and more stable gradient descent convergence** during training.